In [ ]:
import os
import numpy as np
import random
import warnings
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error, r2_score
from skopt import BayesSearchCV
from skopt.space import Real, Integer

os.environ['PYTHONHASHSEED'] = str(1)
np.random.seed(1)
random.seed(1)
warnings.filterwarnings("ignore")

FILENAME = 'cleaned_data_perfect.xlsx'
TARGET_COL = '好氧池北溶解氧'

def load_data(filename):
    if not os.path.exists(filename):
        raise FileNotFoundError(f"{filename} not found.")
    data = pd.read_excel(filename)
    if TARGET_COL not in data.columns:
        raise ValueError(f"Target {TARGET_COL} not found.")
    data = data.dropna(subset=[TARGET_COL])
    return data

def feature_engineering(data):
    if '时间' in data.columns:
        data = data.drop(columns=['时间'])
    
    cols_to_drop = ['E', 'V', 'S']
    existing = [c for c in cols_to_drop if c in data.columns]
    data = data.drop(columns=existing)
    return data

def preprocess_and_split(X, y, shuffle=True):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=1, shuffle=shuffle
    )

    imputer = KNNImputer(n_neighbors=5)
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    pt = PowerTransformer(method='yeo-johnson')
    X_train_pt = pt.fit_transform(X_train_imp)
    X_test_pt = pt.transform(X_test_imp)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train_pt)
    X_test_sc = scaler.transform(X_test_pt)

    X_train_final = pd.DataFrame(X_train_sc, columns=X.columns)
    X_test_final = pd.DataFrame(X_test_sc, columns=X.columns)

    artifacts = {'imputer': imputer, 'pt': pt, 'scaler': scaler}
    return X_train_final, X_test_final, y_train, y_test, artifacts

def optimize_model(X_train, y_train):
    model = RandomForestRegressor(random_state=1, n_jobs=-1)

    search_space = {
        'max_depth': Integer(3, 15),
        'max_features': Real(0.1, 0.9),
        'min_samples_leaf': Integer(1, 10),
        'min_samples_split': Integer(2, 20),
        'n_estimators': Integer(100, 500)
    }

    optimizer = BayesSearchCV(
        estimator=model,
        search_spaces=search_space,
        n_iter=30,
        scoring='neg_mean_squared_error',
        cv=5,
        random_state=1,
        n_jobs=-1
    )

    optimizer.fit(X_train, y_train)
    return optimizer.best_estimator_, optimizer.best_params_

def main():
    try:
        raw_data = load_data(FILENAME)
        data = feature_engineering(raw_data)

        X = data.drop(columns=[TARGET_COL])
        y = data[TARGET_COL].reset_index(drop=True)

        X_train, X_test, y_train, y_test, artifacts = preprocess_and_split(X, y, shuffle=True)

        print("Starting Bayesian Optimization...")
        model, best_params = optimize_model(X_train, y_train)
        print(f"Best Params: {dict(best_params)}")

        y_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        print(f"Test RMSE: {rmse:.4f}")
        print(f"Test R2: {r2:.4f}")

        pd.DataFrame({'True': y_test, 'Pred': y_pred}).to_excel('predictions.xlsx', index=False)

        joblib.dump(model, 'rf_model.pkl')
        joblib.dump(artifacts, 'artifacts.pkl')
        print("Process completed successfully.")

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()